# Explorer 05 - Snapshot comparison

**Purpose.** Diff January 2020 against October 2026: which labels changed parent, and how late releases get catalogued.

**Feeds main report section:** 5.4 Snapshot comparison, and 7. Limitations

**Modules used:** `vinyl.snapshots`, `vinyl.labels`, `vinyl.viz`

**Tables read:** `labels` and `releases` from both snapshots

This is a working notebook. It can be messy; anything that goes into the report is re-run from `main.ipynb` through the same functions in `vinyl/`.

In [ ]:
# Make the `vinyl` package importable from the explorers/ folder.
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "explorers":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

%load_ext autoreload
%autoreload 2

from pyspark.sql import functions as F
from vinyl import config, ingest

spark = ingest.get_spark(app_name="vinyl-05-snapshots")
from vinyl import snapshots, viz

In [ ]:
old, new = config.BASE_SNAPSHOT, config.MAIN_SNAPSHOT   # "2020", "2026"
labels_old, labels_new = ingest.read_table(spark, old, "labels"), ingest.read_table(spark, new, "labels")
releases_old, releases_new = ingest.read_table(spark, old, "releases"), ingest.read_table(spark, new, "releases")

## 1. Labels whose parent changed

In [ ]:
changes = snapshots.label_parent_changes(labels_old, labels_new).cache()
changes.groupBy("change").count().orderBy(F.desc("count")).show()

In [ ]:
# Biggest ownership changes: rank changed labels by how many in-scope releases they have.
release_labels = ingest.read_table(spark, new, "release_labels")
volume = release_labels.groupBy("label_id").agg(F.countDistinct("release_id").alias("n_releases"))
(changes.where(F.col("change").isin("changed_parent", "gained_parent", "lost_parent"))
    .join(volume, "label_id").orderBy(F.desc("n_releases")).show(30, truncate=False))

In [ ]:
# TODO: which of these moved INTO or OUT OF a major tree? Resolve roots in both snapshots
#       (labels.resolve_label_roots + tag_major_labels) and compare `major` old vs new.
# TODO: hand-check the top cases: real acquisition, or a data correction?

## 2. Cataloguing lag

In [ ]:
lag = snapshots.cataloguing_lag(releases_old, releases_new)
lag_pd = lag.toPandas()
lag_pd

In [ ]:
viz.plot_lines(lag_pd[lag_pd.year <= 2019], "year", "completeness",
               title="TODO: share of each year's releases already catalogued by Jan 2020", percent=True)

## 3. Estimated undercount of recent years

In [ ]:
yearly = releases_new.groupBy("year").agg(F.count("*").alias("n_releases"))
snapshots.estimate_undercount(lag, yearly).toPandas()

In [ ]:
# TODO: 2026 is a nine-month year in the October snapshot - correct or exclude.
# TODO: does lag differ by format (vinyl vs file) or genre? If so, recent-year SHARES are biased too.

## Decisions to carry into `main.ipynb`

Fill this in when the exploration is done. Each line should end up as a sentence in the report's methodology or limitations.

- [ ] List of confirmed ownership changes worth naming in the report.
- [ ] Completeness by age (the correction table) and which years are flagged as undercounted.
- [ ] Whether trend charts stop at 2024 or show 2025-2026 with a caveat.